# FraudTrace AI - 00: Colab Environment Verification & Smoke Test

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Verify Google Colab environment, dependencies, dataset presence, and execute a smoke test pipeline.

| Attribute | Description |
| :--- | :--- |
| **Input** | Git repository clone, requirements-colab.txt, ml/data/sample/ |
| **Output** | Verified environment report and passed smoke test execution |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Environment & Hardware Diagnostics
We inspect the Python runtime, OS platform, and check for available accelerator devices (GPU/CPU).

In [ ]:
import platform
import sys
import sklearn
import numpy as np
import pandas as pd

print("=== Environment Diagnostics ===")
print(f"Python Version  : {platform.python_version()}")
print(f"Platform        : {platform.platform()}")
print(f"NumPy Version   : {np.__version__}")
print(f"Pandas Version  : {pd.__version__}")
print(f"Scikit-Learn    : {sklearn.__version__}")

try:
    import torch
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "None (CPU Mode)"
    print(f"PyTorch CUDA    : {cuda_available} ({device_name})")
except ImportError:
    print("PyTorch CUDA    : Not installed (Running in Scikit-Learn standard CPU mode)")


## 2. Verify Directory Structure & Sample Datasets
Check that relative paths exist and sample data files are accessible.

In [ ]:
from ml.src.config import get_data_dir, get_artifacts_dir
from ml.src.data.loader import DataLoader

sample_dir = get_data_dir("sample")
artifacts_dir = get_artifacts_dir()

print(f"Sample Data Path: {sample_dir}")
print(f"Artifacts Path  : {artifacts_dir}")

assert sample_dir.exists(), f"Sample data directory not found at {sample_dir}"

loader = DataLoader(dataset_type="sample")
golden_tests = loader.load_golden_tests()
print(f"[OK] Successfully loaded {len(golden_tests)} golden test cases from sample suite.")


## 3. End-to-End Pipeline Smoke Test
Execute the `ForensicInferencePipeline` on a sample pairwise conflict test.

In [ ]:
from ml.src.inference.pipeline import ForensicInferencePipeline

pipeline = ForensicInferencePipeline()

sample_evidence = [
    {
        "id": "EV-SMOKE-1",
        "type": "FIR",
        "source": "Complainant Statement",
        "amount": 50000.0,
        "timestamp": "2024-03-15T10:30:00Z",
        "entities": [{"type": "phone", "value": "+91 98765-43210"}]
    },
    {
        "id": "EV-SMOKE-2",
        "type": "BANK_STATEMENT",
        "source": "HDFC Bank Core",
        "amount": 48000.0,
        "timestamp": "2024-03-15T10:32:00Z",
        "entities": [{"type": "phone", "value": "09876543210"}]
    }
]

result = pipeline.analyze_event(sample_evidence)
print("=== Smoke Test Result ===")
print(f"Contradiction Status: {result['contradiction']['status']}")
print(f"Confidence Level    : {result['confidence']['level']} (Score: {result['confidence']['score']})")
print(f"Discrepancies Found : {len(result['discrepancies'])}")
print("[OK] Smoke test completed successfully!")
